In [ ]:
from pathlib import Path

import pandas as pd


DATA_DIR = Path.cwd()
FILES = {
    "mtrain2026-o.csv": "mtrain2026.csv",
    "mtest2026-o.csv": "mtest2026.csv",
    "wtrain2026-o.csv": "wtrain2026.csv",
    "wtest2026-o.csv": "wtest2026.csv",
}

TEAM_PREFIX = "diff_TeamAvg"
OPP_PREFIX = "diff_Opp"
DERIVED_LAST = [
    "diff_SeedNum",
    "diff_DefensiveEfficiency",
    "diff_OffensiveEfficiency",
    "diff_Tempo",
    "diff_conf_eigen_rating",
    "diff_conf_massey_rating",
    "diff_eigen_rating",
    "diff_massey_rating",
    "rating_diff_x_tempo_diff",
]
PREFERRED_DUPLICATES = {
    "tempo_diff": "diff_Tempo",
}


def build_id(df: pd.DataFrame) -> pd.Series:
    return (
        df["Season"].astype(str)
        + "_"
        + df["Team1ID"].astype(str)
        + "_"
        + df["Team2ID"].astype(str)
    )


def find_duplicate_stat_columns(df: pd.DataFrame, stat_cols: list[str]) -> list[str]:
    columns_to_drop = set()

    for duplicate_col, preferred_col in PREFERRED_DUPLICATES.items():
        if (
            duplicate_col in stat_cols
            and preferred_col in stat_cols
            and df[duplicate_col].equals(df[preferred_col])
        ):
            columns_to_drop.add(duplicate_col)

    remaining = [col for col in stat_cols if col not in columns_to_drop]
    for i, left in enumerate(remaining):
        for right in remaining[i + 1 :]:
            if df[left].equals(df[right]):
                columns_to_drop.add(right)

    return sorted(columns_to_drop)


def ordered_stat_columns(df: pd.DataFrame, metadata_cols: list[str]) -> list[str]:
    stat_cols = [col for col in df.columns if col not in metadata_cols]
    duplicate_cols = find_duplicate_stat_columns(df, stat_cols)
    stat_cols = [col for col in stat_cols if col not in duplicate_cols]

    team_cols = sorted(col for col in stat_cols if col.startswith(TEAM_PREFIX))
    opp_cols = sorted(col for col in stat_cols if col.startswith(OPP_PREFIX))
    derived_cols = [col for col in stat_cols if col not in set(team_cols + opp_cols)]

    derived_cols = [col for col in DERIVED_LAST if col in derived_cols] + sorted(
        col for col in derived_cols if col not in DERIVED_LAST
    )

    return team_cols + opp_cols + derived_cols


def clean_train(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.insert(0, "ID", build_id(cleaned))

    metadata_cols = ["ID", "Team1Name", "Team2Name", "Team1Win"]
    stat_cols = ordered_stat_columns(
        cleaned,
        metadata_cols + ["Season", "Team1ID", "Team2ID"],
    )

    return cleaned[metadata_cols + stat_cols]


def clean_test(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    metadata_cols = ["ID", "Team1Name", "Team2Name"]
    stat_cols = ordered_stat_columns(cleaned, metadata_cols)

    return cleaned[metadata_cols + stat_cols]


for old_name, new_name in FILES.items():
    path = DATA_DIR / old_name
    df = pd.read_csv(path)

    if "Team1Win" in df.columns:
        cleaned = clean_train(df)
    else:
        cleaned = clean_test(df)

    cleaned.to_csv(DATA_DIR / new_name, index=False)

    print(f"{old_name} -> {new_name}")
    print(cleaned.columns.tolist())
    print()
